# Crema & Ferro: parla con l'assistente da Python / Talk to the assistant from Python

**IT** Questo quaderno chiama l'API pubblica della demo (https://chat-bot.giuseppebosi.com/tutorial) e stampa la risposta man mano che arriva (*streaming*). Premi **Esegui** (▶) sulla cella qui sotto: non serve installare niente. Colab ha già `requests`.

**EN** This notebook calls the demo's public API and prints the reply as it streams in. Press **Run** (▶) on the cell below: nothing to install, Colab already has `requests`.

In [ ]:
import json
import requests

URL = "https://chat-bot.giuseppebosi.com/api/chat"
HEADERS = {
    "Content-Type": "application/json",
    "x-cf-key": "cfdemo_pub_2026",      # public key of the demo (printed in /tutorial)
    "User-Agent": "chat-bot-notebook",  # always name your client: some APIs refuse requests without it
}

def ask(question, chat_id="notebook"):
    body = {"id": chat_id, "messages": [{"id": "u1", "role": "user",
            "parts": [{"type": "text", "text": question}]}]}
    with requests.post(URL, headers=HEADERS, json=body, stream=True, timeout=120) as r:
        r.raise_for_status()
        r.encoding = "utf-8"  # a stream has no declared charset: without this, accents break
        for line in r.iter_lines(decode_unicode=True):
            if not line.startswith("data: ") or line == "data: [DONE]":
                continue
            event = json.loads(line[6:])
            if event["type"] == "tool-input-available":
                print(f"[attrezzo / tool: {event['toolName']}]")
            elif event["type"] == "text-delta":
                print(event["delta"], end="", flush=True)
    print()

ask("Quanto costa la spedizione?")

**Cosa dovresti vedere / What you should see:** una riga `[attrezzo / tool: searchHelp]` (l'assistente ha letto il centro assistenza) e poi la risposta in italiano, con i 6,90 € di spedizione e la spedizione gratuita da 49 €.

Prova un'altra domanda / Try another question:

In [ ]:
ask("Consigliami una moka per due persone, massimo 40 euro")